# Final Model Code


In [1]:
import numpy as np
import polars as pl
from catboost import CatBoostRegressor, Pool

## Feature Engineering Functions

In [2]:
def create_time_features(df):
    """Create time-based features"""
    df = df.with_columns([
        pl.col('datetime').dt.hour().alias('hour'),
        pl.col('datetime').dt.day().alias('day'),
        pl.col('datetime').dt.month().alias('month'),
        pl.col('datetime').dt.weekday().alias('weekday'),
        (np.sin(pl.col('datetime').dt.hour() * 2 * np.pi / 24)).alias('hour_sin'),
        (np.cos(pl.col('datetime').dt.hour() * 2 * np.pi / 24)).alias('hour_cos'),
        (np.sin(pl.col('datetime').dt.day() * 2 * np.pi / 31)).alias('day_sin'),
        (np.cos(pl.col('datetime').dt.day() * 2 * np.pi / 31)).alias('day_cos'),
        (np.sin(pl.col('datetime').dt.month() * 2 * np.pi / 12)).alias('month_sin'),
        (np.cos(pl.col('datetime').dt.month() * 2 * np.pi / 12)).alias('month_cos'),
        (np.sin(pl.col('datetime').dt.weekday() * 2 * np.pi / 7)).alias('weekday_sin'),
        (np.cos(pl.col('datetime').dt.weekday() * 2 * np.pi / 7)).alias('weekday_cos'),
    ])
    return df
def create_lag_features(df, target_col='station1_PM2.5', lags=[1, 2, 3, 6, 12, 24]):
    """Create lag features"""
    for lag in lags:
        df = df.with_columns(
            pl.col(target_col).shift(lag).alias(f'{target_col}_lag{lag}')
        )
    return df
def create_rolling_features(df, target_col='station1_PM2.5'):
    """Create rolling statistics features"""
    df = df.with_columns([
        pl.col(target_col).rolling_mean(window_size=3).shift(1).alias(f'{target_col}_roll_mean_3'),
        pl.col(target_col).rolling_mean(window_size=6).shift(1).alias(f'{target_col}_roll_mean_6'),
        pl.col(target_col).rolling_mean(window_size=12).shift(1).alias(f'{target_col}_roll_mean_12'),
        pl.col(target_col).rolling_mean(window_size=24).shift(1).alias(f'{target_col}_roll_mean_24'),
        pl.col(target_col).rolling_std(window_size=3).shift(1).alias(f'{target_col}_roll_std_3'),
        pl.col(target_col).rolling_std(window_size=6).shift(1).alias(f'{target_col}_roll_std_6'),
        pl.col(target_col).rolling_std(window_size=12).shift(1).alias(f'{target_col}_roll_std_12'),
        pl.col(target_col).rolling_std(window_size=24).shift(1).alias(f'{target_col}_roll_std_24'),
        pl.col(target_col).rolling_max(window_size=24).shift(1).alias(f'{target_col}_roll_max_24'),
        pl.col(target_col).rolling_min(window_size=24).shift(1).alias(f'{target_col}_roll_min_24'),
    ])
    return df

## Model Training

In [3]:
# Load training data (use all rows from df-train.csv)
df_train = pl.read_csv('data/df-train.csv')

# Convert datetime column if needed
if df_train['datetime'].dtype == pl.Utf8:
    df_train = df_train.with_columns(pl.col('datetime').str.to_datetime())

# Cast numeric columns to Float64; strings like 'NA' become null automatically
numeric_cols = [col for col in df_train.columns if col != 'datetime']
cast_exprs = []
for col in numeric_cols:
    cast_exprs.append(pl.col(col).cast(pl.Float64, strict=False).alias(col))
if cast_exprs:
    df_train = df_train.with_columns(cast_exprs)

# Feature engineering
df_train_fe = create_time_features(df_train)
df_train_fe = create_lag_features(df_train_fe)
df_train_fe = create_rolling_features(df_train_fe)

# Drop nulls (baseline data for training and feature generation)
df_train_clean = df_train_fe.drop_nulls()

# Define target and feature list
target = 'station1_PM2.5'
exclude_cols = ['datetime', target]
selected_features = [col for col in df_train_clean.columns if col not in exclude_cols]

print(f"Train shape: {df_train_clean.shape}")
print(f"Feature count: {len(selected_features)}")

# Prepare training arrays
X_train = df_train_clean.select(selected_features).to_numpy()
y_train = df_train_clean.get_column(target).to_numpy()

# CatBoost Pool
train_pool = Pool(data=X_train, label=y_train, feature_names=selected_features)

# Fixed final hyperparameters
best_params = {
    'iterations': 938,
    'learning_rate': 0.061,
    'depth': 4,
    'l2_leaf_reg': 3.311,
    'random_strength': 0.376,
    'bagging_temperature': 0.408
}
print(f"Using tuned hyperparameters: {best_params}")

final_model = CatBoostRegressor(
    iterations=int(best_params['iterations']),
    learning_rate=float(best_params['learning_rate']),
    depth=int(best_params['depth']),
    l2_leaf_reg=float(best_params['l2_leaf_reg']),
    random_strength=float(best_params['random_strength']),
    bagging_temperature=float(best_params['bagging_temperature']),
    loss_function='MAE',
    eval_metric='MAE',
    verbose=100,
    random_seed=42
)

# Train final model (full training set)
print("Start training final model...")
final_model.fit(train_pool)
print("Model training complete!")

Train shape: (11104, 53)
Feature count: 51
Using tuned hyperparameters: {'iterations': 938, 'learning_rate': 0.061, 'depth': 4, 'l2_leaf_reg': 3.311, 'random_strength': 0.376, 'bagging_temperature': 0.408}
Start training final model...
0:	learn: 8.5202296	total: 133ms	remaining: 2m 4s
100:	learn: 1.5578090	total: 317ms	remaining: 2.62s
200:	learn: 1.3805179	total: 493ms	remaining: 1.81s
300:	learn: 1.2880285	total: 672ms	remaining: 1.42s
400:	learn: 1.2292304	total: 853ms	remaining: 1.14s
500:	learn: 1.1820651	total: 1.03s	remaining: 903ms
600:	learn: 1.1404151	total: 1.25s	remaining: 699ms
700:	learn: 1.1113412	total: 1.44s	remaining: 486ms
800:	learn: 1.0851198	total: 1.61s	remaining: 275ms
900:	learn: 1.0603432	total: 1.8s	remaining: 73.8ms
937:	learn: 1.0531114	total: 1.87s	remaining: 0us
Model training complete!


## Prediction

In [4]:
# Read forecast data 
df_forecast = pl.read_csv('data/df-forecast.csv')

# Convert datetime column if needed
if df_forecast['datetime'].dtype == pl.Utf8:
    df_forecast = df_forecast.with_columns(pl.col('datetime').str.to_datetime())

print(f"Forecast data shape: {df_forecast.shape}")
print(f"Forecast columns: {df_forecast.columns}")

# Use last clean training row to fill missing feature columns in forecast data
df_train_last = df_train_clean.tail(1)

df_forecast_full = df_forecast.clone()
for col in df_train_clean.columns:
    if col not in df_forecast_full.columns and col != 'datetime':
        if col != target:
            last_value = df_train_last[col][0]
            df_forecast_full = df_forecast_full.with_columns(pl.lit(last_value).alias(col))
        else:
            df_forecast_full = df_forecast_full.with_columns(pl.lit(None).alias(col))

# Align column order with training data
df_forecast_full = df_forecast_full.select(df_train_clean.columns)

# Cast numeric columns to Float64 for both train/forecast before concat
df_train_for_concat = df_train_clean
cast_exprs_train = []
cast_exprs_forecast = []
for col, dtype in zip(df_train_clean.columns, df_train_clean.dtypes):
    if col != 'datetime':
        cast_exprs_train.append(pl.col(col).cast(pl.Float64, strict=False).alias(col))
        cast_exprs_forecast.append(pl.col(col).cast(pl.Float64, strict=False).alias(col))
if cast_exprs_train:
    df_train_for_concat = df_train_clean.with_columns(cast_exprs_train)
if cast_exprs_forecast:
    df_forecast_full = df_forecast_full.with_columns(cast_exprs_forecast)

# Combine train + forecast for lag/rolling feature generation
df_combined = pl.concat([df_train_for_concat, df_forecast_full])
print(f"Combined data shape: {df_combined.shape}")

Forecast data shape: (24, 2)
Forecast columns: ['datetime', 'station1_PM2.5']
Combined data shape: (11128, 53)


In [5]:
# Apply feature engineering
df_combined = create_time_features(df_combined)
df_combined = create_lag_features(df_combined)
df_combined = create_rolling_features(df_combined)

# Forecast slice starts after the clean training rows
forecast_start_idx = len(df_train_clean)
df_forecast_fe = df_combined[forecast_start_idx:]

In [6]:
# Recursive forecasting
predictions = []
# Compute feature means from training data for filling nulls
train_feature_stats = df_train_clean.select(selected_features).mean()
feature_means = {}
for col in selected_features:
    if col in train_feature_stats.columns:
        mean_val = train_feature_stats[col][0]
        feature_means[col] = mean_val if mean_val is not None else 0.0
    else:
        feature_means[col] = 0.0

for i in range(df_forecast_fe.height):
    # Prepare features for current step
    X_pred = df_forecast_fe[i:i+1].select(selected_features)
    # Fill missing values per feature
    fill_exprs = []
    for col in selected_features:
        if col in X_pred.columns:
            fill_val = feature_means.get(col, 0.0)
            fill_exprs.append(pl.col(col).forward_fill().fill_null(fill_val).alias(col))
    
    X_pred_filled = X_pred.with_columns(fill_exprs) if fill_exprs else X_pred
    X_pred_np = np.nan_to_num(X_pred_filled.to_numpy(), nan=0.0)
    pred = final_model.predict(X_pred_np)[0]
    predictions.append(float(pred))
    current_datetime = df_forecast_fe[i]['datetime'][0]
    df_combined = df_combined.with_columns(
        pl.when(pl.col('datetime') == current_datetime)
        .then(pred)
        .otherwise(pl.col(target))
        .alias(target)
    )
    df_combined = create_lag_features(df_combined, target_col=target)
    df_combined = create_rolling_features(df_combined, target_col=target)
    df_forecast_fe = df_combined[forecast_start_idx:]
    
    if (i + 1) % 6 == 0:
        print(f"Predicted {i + 1}/{df_forecast_fe.height}")

Predicted 6/24
Predicted 12/24
Predicted 18/24
Predicted 24/24


In [7]:
# Write predictions back to df-forecast.csv (keep original column names)
# Ensure prediction length matches df_forecast
if len(predictions) != df_forecast.height:
    raise ValueError(f"Prediction count ({len(predictions)}) != df_forecast rows ({df_forecast.height}). Stop writing.")

predictions_series = pl.Series('station1_PM2.5', predictions, dtype=pl.Float64)
df_forecast = df_forecast.with_columns(predictions_series)

# Save results (only datetime and station1_PM2.5)
df_forecast.select(['datetime', 'station1_PM2.5']).write_csv('data/df-forecast.csv')

print("Prediction finished!")

Prediction finished!
